# PROJECT NOIR ? The Nishidak Broadcast
### Colab workprint renderer for the current 05:52 NOIR timeline
**Visual direction:** Sayan ? **Score:** `Noir.wav` by Atandra

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayan-bhattacharya/noir-analog-horror/blob/main/colab_noir_accelerator.ipynb)

This notebook clones the separate `noir-analog-horror` repository and renders its current timeline. The master screenplay and score are 30:22, while the implemented compositor currently ends at 05:52; this notebook renders that available workprint and does not imply the remaining runtime is implemented. It never substitutes podcast audio for `Noir.wav`.

Shot 30 includes slow courtyard-cloud motion, a mostly steady analog image, the purple charged-particle portal, and Bimal?s slow approach, slip, and fall.


## 1. Hardware Detection & Verification


In [ ]:
import torch
if torch.cuda.is_available():
    print(f"[SUCCESS] GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[INFO] Running on Standard CPU. For fast renders, select Runtime -> Change runtime type -> T4 GPU")

## 1. Clone the standalone NOIR repository and install render dependencies


In [ ]:
import os, sys, subprocess
from pathlib import Path

project_dir = Path("/content/noir-analog-horror")
if not project_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/sayan-bhattacharya/noir-analog-horror.git", str(project_dir)], check=True)
else:
    print("Using existing NOIR checkout; no files will be deleted or reset.")
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
os.environ["PYTHONPATH"] = str(project_dir)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "soundfile", "scipy", "pillow", "uharfbuzz", "freetype-py", "pyyaml"], check=True)

# Use only the original NOIR score. Drive is optional; upload Noir.wav if Drive cannot mount.
from google.colab import drive, files
drive_mounted = False
try:
    drive.mount("/content/drive", timeout_ms=60000)
    drive_mounted = True
except Exception as exc:
    print(f"Google Drive did not mount ({type(exc).__name__}). Continuing with file upload if needed.")

score_candidates = []
if drive_mounted:
    score_candidates.append(Path("/content/drive/MyDrive/NOIR_Film/Noir.wav"))
score_candidates.extend([project_dir / "Noir.wav", project_dir / "audio" / "Noir.wav"])
score_path = next((candidate for candidate in score_candidates if candidate.is_file()), None)
if score_path is None:
    print("Upload the original Noir.wav score (not a podcast mix).")
    uploaded = files.upload()
    uploaded_score = next((Path(name) for name in uploaded if Path(name).name.lower() == "noir.wav"), None)
    if uploaded_score is not None and uploaded_score.is_file():
        score_path = uploaded_score
if score_path is None or not score_path.is_file():
    raise FileNotFoundError("Noir.wav is required. No podcast or other audio will be substituted.")
project_score = project_dir / "Noir.wav"
if score_path.resolve() != project_score.resolve():
    import shutil
    shutil.copy2(score_path, project_score)
print(f"NOIR score ready: {project_score}")


## 2. Render the current NOIR workprint
The renderer currently covers `00:00?05:52` (352 seconds). The 30:22 screenplay has not yet been fully implemented in the compositor. The default is a quick 640?360, 12 fps draft. Set `RENDER_PREVIEW = False` in the next cell for the 1280?720, 24 fps final-quality workprint.


In [ ]:
import os, sys, time, subprocess
from pathlib import Path

os.chdir("/content/noir-analog-horror")
os.environ["PYTHONPATH"] = "."
RENDER_SECONDS = 352
RENDER_PREVIEW = True  # False = 1280x720 at 24 fps
RENDER_QUALITY = "preview" if RENDER_PREVIEW else "final"
cmd = [sys.executable, "-m", "noir_film.render_noir", "--duration", str(RENDER_SECONDS), "--quality", RENDER_QUALITY]
if RENDER_PREVIEW:
    cmd.append("--preview")
print("Rendering current NOIR workprint:", " ".join(cmd))
start_time = time.time()
subprocess.run(cmd, check=True)
print(f"Render complete in {time.time() - start_time:.1f}s")


## 4. In-Notebook Video Player


In [ ]:
from IPython.display import Video, display
from pathlib import Path
import glob, os
matches = glob.glob("/content/noir-analog-horror/output/noir_master_352s_*.mp4")
if not matches:
    raise FileNotFoundError("NOIR render not found. Run the render cell first.")
video_path = max(matches, key=os.path.getmtime)
print("Playing:", Path(video_path).name)
display(Video(video_path, embed=False, width=900))


## 5. Download Master Film (720p HD MP4)


In [ ]:
from google.colab import files
from pathlib import Path
import glob, os
matches = glob.glob("/content/noir-analog-horror/output/noir_master_352s_*.mp4")
if not matches:
    raise FileNotFoundError("NOIR render not found. Run the render cell first.")
video_path = max(matches, key=os.path.getmtime)
print("Downloading:", Path(video_path).name)
files.download(video_path)


## 4. Save the NOIR workprint to Google Drive


In [ ]:
import shutil, glob, os
from pathlib import Path
from google.colab import drive, files
matches = glob.glob("/content/noir-analog-horror/output/noir_master_352s_*.mp4")
if not matches:
    raise FileNotFoundError("NOIR render not found. Run the render cell first.")
src = Path(max(matches, key=os.path.getmtime))
try:
    drive.mount("/content/drive", timeout_ms=60000)
    dest_dir = Path("/content/drive/MyDrive/NOIR_Film/renders")
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / src.name
    shutil.copy2(src, dest)
    print(f"Saved NOIR workprint to Drive: {dest}")
except Exception as exc:
    print(f"Drive save failed ({type(exc).__name__}). Starting a direct browser download instead.")
    files.download(str(src))
